# Imports

In [10]:
import torch
from torch.nn import CrossEntropyLoss
from segmentation_models_pytorch.losses import DiceLoss
from torch.utils.data import DataLoader
from torchvision import transforms
from RuralDataset import RuralDataset
from SegmentationModel import SegmentationModel
from Trainer import Trainer
from Evaluator import Evaluator

torch.cuda.empty_cache()

# Configuration parameters

In [11]:
DATA_ROOT = 'train'
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
NUM_CLASSES = 9
MODEL_SAVE_PATH = 'saved_model.pth'

# Device configuration

In [12]:
if torch.cuda.is_available():
    device = torch.device('cuda')  # Configura per utilizzare la GPU
    print(f"Utilizzo GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')  # Fallback alla CPU
    print("CUDA non disponibile, utilizzo CPU")

Utilizzo GPU: NVIDIA GeForce RTX 5070 Ti


# Transforms initialization

In [13]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(512, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05), shear=5),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.3),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dataset loading and splitting into training and validation sets

In [14]:
from logging import root
# Crea dataset separati con le rispettive trasformazioni
from torch.utils.data import random_split, Subset

# Crea il dataset originale senza augmentation per ottenere gli indici di split
original_dataset = RuralDataset(root_dir=DATA_ROOT)

# Suddividi gli indici del dataset originale in train e val
train_size = int(0.8 * len(original_dataset))
val_size = len(original_dataset) - train_size

train_dataset_indices, val_dataset_indices = random_split(original_dataset, [train_size, val_size])

# Ora crea i dataset effettivi applicando augmentation e trasformazioni
# Crea il dataset di training con augmentation e la trasformazione di training, usando gli indici di training
train_dataset_augmented = RuralDataset(root_dir=DATA_ROOT, transform=train_transform, augment=True)

# Crea il dataset di validazione senza augmentation e con la trasformazione di validazione, usando gli indici di validazione
val_dataset = RuralDataset(root_dir=DATA_ROOT, transform=val_transform, augment=False)
val_subset = Subset(val_dataset, val_dataset_indices.indices)

train_loader = DataLoader(
    train_dataset_augmented, batch_size=BATCH_SIZE, num_workers=16, shuffle=True,
    pin_memory=True)
val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, num_workers=16, shuffle=False,
    pin_memory=True)

print(f"Training on {len(train_loader.dataset)} samples, validating on {len(val_loader.dataset)} samples.")

Training on 2979 samples, validating on 745 samples.


# Model initialization

In [15]:
model = SegmentationModel(NUM_CLASSES)
model.to(device)
print("Model architecture:")
print(model)

Model architecture:
SegmentationModel(
  (model): Unet(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=F

# Loss Function and optimizer definition

In [16]:
# weights = Trainer.compute_class_weights(train_loader, num_classes=NUM_CLASSES)
weights = [ 3.8668,  1.0843,  0.7316,  0.7333,  0.8368,  0.3093,  1.7578, 62.0476, 13.2180]

cross_entropy = CrossEntropyLoss(weight=torch.tensor(weights).to(device))
dice_loss = DiceLoss(mode='multiclass')

def combined_loss(pred, target):
    return cross_entropy(pred, target) + dice_loss(pred, target)

criterion = combined_loss
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


# Trainer initialization and training

In [ ]:
trainer = Trainer(model, train_loader, val_loader, criterion, optimizer, device)
print("\nStarting training...")
trainer.run(num_epochs=NUM_EPOCHS, model_save_path=MODEL_SAVE_PATH)
print("Training finished.")


Starting training...

Epoch 1/50
Batch 1/373, Loss: 3.3113
Batch 2/373, Loss: 3.1108
Batch 3/373, Loss: 3.3708
Batch 4/373, Loss: 2.9944
Batch 5/373, Loss: 3.1943
Batch 6/373, Loss: 2.7525
Batch 7/373, Loss: 3.0833
Batch 8/373, Loss: 3.0479
Batch 9/373, Loss: 2.8996
Batch 10/373, Loss: 3.0489
Batch 11/373, Loss: 3.1212
Batch 12/373, Loss: 2.8498
Batch 13/373, Loss: 2.8588
Batch 14/373, Loss: 2.7196
Batch 15/373, Loss: 2.9352
Batch 16/373, Loss: 2.8426
Batch 17/373, Loss: 2.6737
Batch 18/373, Loss: 2.8832
Batch 19/373, Loss: 2.8431
Batch 20/373, Loss: 2.8789
Batch 21/373, Loss: 2.8048
Batch 22/373, Loss: 2.8158
Batch 23/373, Loss: 2.6095
Batch 24/373, Loss: 2.6234
Batch 25/373, Loss: 2.6128
Batch 26/373, Loss: 2.7268
Batch 27/373, Loss: 2.7482
Batch 28/373, Loss: 2.7606
Batch 29/373, Loss: 2.6659
Batch 30/373, Loss: 2.7249
Batch 31/373, Loss: 2.6117
Batch 32/373, Loss: 2.6056
Batch 33/373, Loss: 2.5175
Batch 34/373, Loss: 2.4581
Batch 35/373, Loss: 2.6281
Batch 36/373, Loss: 2.4867
Bat

# Model evaluation

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH))

evaluator = Evaluator(model, val_loader, device)
metrics = evaluator.evaluate_classification_metrics()

# Example Predict

In [ ]:
evaluator.predict_from_folder(folder_number = 153)

In [ ]:
print(metrics.get('accuracy', 'N/A'))
print(metrics.get('f1_score', 'N/A'))
print(metrics.get('precision', 'N/A'))
print(metrics.get('recall', 'N/A'))